# Lab 4 · Generate the ontology **from your data**

Instead of hand-designing an ontology, this notebook **reads your `gold` and `silver` tables and generates the
ontology blueprint for you** — the entities, their keys and table bindings, and the relationships that connect them,
with the *exact* source columns to use.

You then apply this blueprint in the Ontology editor (a short, mechanical checklist — no modelling decisions) and
build the data agent on top. Run **all** cells, then keep the printed **blueprint** on screen for the next task.

> Make sure the **`lh_resident360`** Lakehouse is attached and you've completed Lab 1 (so `gold.resident_360` and the
> `silver.fact_*` tables exist).

## 0 · Setup

In [ ]:
from pyspark.sql import functions as F

# The tables the ontology is generated from (all built in Lab 1)
GOLD_360   = "gold.resident_360"
FACT_EVENT = "silver.fact_event_attendance"
FACT_PROG  = "silver.fact_programme_enrolment"
FACT_CHAL  = "silver.fact_challenge"

print("Reading from:", GOLD_360, FACT_EVENT, FACT_PROG, FACT_CHAL)

## 1 · Discover the entities

An **entity** is a real-world thing with a stable key. We scan the tables for key-like columns (`*_id` or a low-
cardinality category such as `region`) and count how many distinct instances each has.

In [ ]:
# Candidate entity keys: (Entity name, source table, key column)
candidates = [
    ("Resident",  GOLD_360,   "resident_id"),
    ("Region",    GOLD_360,   "region"),
    ("Event",     FACT_EVENT, "event_id"),
    ("Programme", FACT_PROG,  "programme_name"),
    ("Challenge", FACT_CHAL,  "challenge_name"),
]

entities = []
for name, tbl, key in candidates:
    try:
        n = spark.table(tbl).select(key).where(F.col(key).isNotNull()).distinct().count()
        entities.append((name, tbl, key, n))
        print(f"  {name:10s}  key={key:16s}  binding={tbl:28s}  instances={n}")
    except Exception as e:
        print(f"  [skip] {name}: {e}")

## 2 · Discover the relationships

A **relationship** exists when a table contains the keys of two entities in the same row — that row *is* the link.
We check each fact table for pairs of entity keys and confirm they actually join.

In [ ]:
# (Relationship name, mapping table, origin entity+key, target entity+key)
rel_candidates = [
    ("livesIn",        GOLD_360,   ("Resident","resident_id"), ("Region","region")),
    ("attended",       FACT_EVENT, ("Resident","resident_id"), ("Event","event_id")),
    ("heldIn",         FACT_EVENT, ("Event","event_id"),       ("Region","region")),
    ("enrolledIn",     FACT_PROG,  ("Resident","resident_id"), ("Programme","programme_name")),
    ("participatesIn", FACT_CHAL,  ("Resident","resident_id"), ("Challenge","challenge_name")),
]

relationships = []
for rname, tbl, (oe, ok), (te, tk) in rel_candidates:
    try:
        cols = spark.table(tbl).columns
        if ok in cols and tk in cols:
            pairs = spark.table(tbl).select(ok, tk).where(F.col(ok).isNotNull() & F.col(tk).isNotNull()).distinct().count()
            relationships.append((rname, tbl, oe, ok, te, tk, pairs))
            print(f"  {oe} --{rname}--> {te:10s}  via {tbl:28s}  ({ok} -> {tk})  links={pairs}")
        else:
            print(f"  [skip] {rname}: keys not both present in {tbl}")
    except Exception as e:
        print(f"  [skip] {rname}: {e}")

## 3 · Your generated ontology blueprint

This is the spec to apply in the Ontology editor. **Timestamp = None** for every entity (these are current-state,
non-timeseries bindings). Copy it — the next task is a mechanical apply, no decisions.

In [ ]:
print("="*70)
print(" ENTITIES  (bind each Lakehouse table; Timestamp = None)")
print("="*70)
print(f"  {'Entity':10s} {'Key':16s} {'Binding (Lakehouse table)':30s} {'Timestamp'}")
for name, tbl, key, n in entities:
    print(f"  {name:10s} {key:16s} {tbl:30s} None")

print()
print("="*70)
print(" RELATIONSHIPS  (edge -> Browse available sources -> map both keys)")
print("="*70)
print(f"  {'Relationship':28s} {'Mapping table':28s} {'Origin key -> Target key'}")
for rname, tbl, oe, ok, te, tk, pairs in relationships:
    label = f"{oe} --{rname}--> {te}"
    print(f"  {label:28s} {tbl:28s} {ok} -> {tk}")

# Persist the blueprint so it's easy to refer back to
import json as _json
blueprint = {
    "entities": [{"entity":n,"key":k,"binding":t,"timestamp":"None","instances":c} for n,t,k,c in entities],
    "relationships": [{"name":rn,"origin":oe,"origin_key":ok,"target":te,"target_key":tk,"mapping_table":t,"links":p}
                       for rn,t,oe,ok,te,tk,p in relationships],
}
dbutils_ok = True
try:
    mssparkutils.fs.put("Files/ontology_blueprint.json", _json.dumps(blueprint, indent=2), overwrite=True)
    print("\nSaved blueprint -> Files/ontology_blueprint.json")
except Exception as e:
    print("\n(blueprint not saved to Files:", e, ")")

---

✅ **Blueprint generated.** You now have the exact entities and relationships — derived from your own data, not guessed.

Next (in the lab README): create the **`resident_ontology`** item and apply this blueprint (a short checklist), then
build the **data agent** on it and compare it against the semantic-model agent.